<a href="https://colab.research.google.com/github/SaddamHasanov/whisper-asr-training-pipeline/blob/main/whisper_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from huggingface_hub import login

token = os.getenv("hf_token")
login(token)

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

In [ ]:
!pip install datasets==3.0.0 transformers accelerate evaluate jiwer tensorboard gradio

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="az", task="transcribe")

In [ ]:
from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="az", task="transcribe")

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

In [ ]:
from datasets import load_dataset

fleurs_data = load_dataset("google/fleurs", "az_az")

In [ ]:
fleurs_data

In [ ]:
example = fleurs_data["train"][0]
example

In [ ]:
from datasets import Audio

fleurs_data = fleurs_data.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
import re

def clean_text(batch):
    text = batch["transcription"].lower()
    text = re.sub(r"[^\w\s]", "", text)  # remove punctuation
    text = text.strip()
    batch["transcription"] = text
    return batch

fleurs_data = fleurs_data.map(clean_text)

In [ ]:
fleurs_data["train"][0]

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]

    # Convert audio → input features
    batch["input_features"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    # Convert text → labels
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids

    return batch

In [ ]:
fleurs_data = fleurs_data.map(
    prepare_dataset,
    remove_columns=fleurs_data["train"].column_names,
    num_proc=2
)

In [ ]:
fleurs_data

In [ ]:
model.generation_config.language = "az"
model.generation_config.task = "transcribe"

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []

In [ ]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        batch["attention_mask"] = batch.get("attention_mask")
        return batch

In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

In [ ]:
import evaluate

metric = evaluate.load("wer")

In [ ]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

FULL FINE-TUNING

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/whisper-model-small-trained",  # change to a repo name of your choice
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=50,
    num_train_epochs=2,
    eval_strategy="epoch",
    fp16=True,
    per_device_eval_batch_size=1,
    generation_max_length=128,
    logging_steps=50,
    remove_unused_columns=False,  # required as the PeftModel forward doesn't have the signature of the wrapped model's forward
    label_names=["labels"],  # same reason as above
    # push_to_hub=True,
)

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=fleurs_data["train"],
    eval_dataset=fleurs_data["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

In [ ]:
processor.save_pretrained(training_args.output_dir)

['/content/drive/MyDrive/Colab Notebooks/whisper-model-small-trained/processor_config.json']

In [ ]:
trainer.train()

In [ ]:
# trainer.evaluate(fleurs_data["test"], predict_with_generate=True)

In [ ]:
# kwargs = {
#     "dataset_tags": "google/fleurs_data",
#     "dataset": "fleurs_data",
#     "language": "az",
#     "model_name": "Whisper-Small-AZ",  # a 'pretty' name for our model
#     "finetuned_from": "openai/whisper-small",
#     "tasks": "automatic-speech-recognition",
# }

In [ ]:
# trainer.push_to_hub(**kwargs)

In [ ]:
# from transformers import pipeline
# import gradio as gr

# pipe = pipeline(model="SaddamHasanov213/Whisper-Small-AZ")  # change to "your-username/the-name-you-picked"

# def transcribe(audio):
#     text = pipe(audio)["text"]
#     return text

# iface = gr.Interface(
#     fn=transcribe,
#     inputs=gr.Audio(source="microphone", type="filepath"),
#     outputs="text",
#     title="Whisper Small AZ"
# )

# iface.launch()